# Aggregate Candidate Features - Gold Layer

Aggregates H3 features for expansion candidate trade areas using H3 polyfill.

**Approach:**
1. Load 5-min isochrones for candidate locations
2. Find all H3 cells whose centers fall inside each isochrone (polyfill)
3. Inner join with h3_features_clean
4. Aggregate by candidate
5. Exclude candidates overlapping existing store trade areas

**Note:** This notebook only aggregates features. Predictions are made in `predict_candidate_sales.ipynb`.

**Inputs:**
- `{catalog}.{silver_schema}.candidate_isochrones` - Candidate trade area polygons (5-min drive time)
- `{catalog}.{silver_schema}.h3_features_clean` - Clean H3 features with derived columns
- `{catalog}.{silver_schema}.isochrones_lce` - Existing store trade areas (for exclusion)

**Output:**
- `{catalog}.{gold_schema}.candidates_features_agg` - Aggregated features per candidate (MA only)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("gold_schema", "")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

# Validate required parameters
assert catalog, "Missing required parameter: catalog"
assert silver_schema, "Missing required parameter: silver_schema"
assert gold_schema, "Missing required parameter: gold_schema"

H3_RESOLUTION = 8

# Table names
candidate_isochrones_table = f"{catalog}.{silver_schema}.candidate_isochrones"
h3_features_table = f"{catalog}.{silver_schema}.h3_features_clean"
lce_isochrones_table = f"{catalog}.{silver_schema}.isochrones_lce"
output_table = f"{catalog}.{gold_schema}.candidates_features_agg"

print(f"Candidate isochrones: {candidate_isochrones_table}")
print(f"H3 features: {h3_features_table}")
print(f"LCE isochrones (for exclusion): {lce_isochrones_table}")
print(f"Output: {output_table}")

## Load Candidate Trade Areas

In [ ]:
# Load candidate isochrones (5-min drive time trade areas)
candidate_isochrones = spark.table(candidate_isochrones_table)

# Standardize columns
columns = candidate_isochrones.columns
id_col = next((c for c in columns if c in ['location_id', 'store_number', 'id']), columns[0])

base_candidates = candidate_isochrones.select(
    col(id_col).alias("candidate_id"),
    col("latitude"),
    col("longitude"),
    col("store_type"),
    col("city") if "city" in columns else lit(None).alias("city"),
    col("state") if "state" in columns else lit(None).alias("state"),
    col("drive_time_minutes") if "drive_time_minutes" in columns else lit(5).alias("drive_time_minutes"),
    col("area_sqkm") if "area_sqkm" in columns else lit(None).alias("area_sqkm"),
    col("geometry")
)

print(f"Loaded {base_candidates.count():,} candidate trade areas")
display(base_candidates.limit(5))

## H3 Polyfill Trade Areas

In [ ]:
# H3 Polyfill: Find all H3 cells whose centers fall inside each candidate's trade area
candidate_h3 = base_candidates.withColumn(
    "h3_cell_id",
    explode(expr(f"h3_polyfillash3string(ST_AsBinary(geometry), {H3_RESOLUTION})"))
)

print(f"Trade areas indexed with H3 resolution {H3_RESOLUTION} using polyfill")
print(f"Total H3 cells across all candidates: {candidate_h3.count():,}")

## Join with Clean H3 Features

In [ ]:
# Load clean H3 features
h3_features = spark.table(h3_features_table)
print(f"Loaded clean H3 features with {h3_features.count():,} cells")

# Join candidate H3 cells with clean features (inner join)
candidate_with_features = candidate_h3.join(h3_features, "h3_cell_id", "inner")
print(f"Joined {candidate_with_features.count():,} H3 cells with features")

## Aggregate Features by Candidate

In [ ]:
# Define features to aggregate
poi_cols = ['retail', 'food_drink', 'leisure', 'education', 'healthcare', 'financial', 'tourism', 'transportation']
existing_poi_cols = [c for c in poi_cols if c in candidate_with_features.columns]

demographic_cols = ['population', 'target_demographic_total']
existing_demo_cols = [c for c in demographic_cols if c in candidate_with_features.columns]

activity_cols = ['human_activity_index', 'total_poi_count']
existing_activity_cols = [c for c in activity_cols if c in candidate_with_features.columns]

urbanity_col = 'urbanity' if 'urbanity' in candidate_with_features.columns else None

# Build aggregation expressions
agg_exprs = []

for demo_col in existing_demo_cols:
    agg_exprs.append(F.sum(F.coalesce(col(demo_col), lit(0))).cast("long").alias(demo_col))

for poi_col in existing_poi_cols:
    agg_exprs.append(F.sum(F.coalesce(col(poi_col), lit(0))).cast("long").alias(poi_col))

if 'total_poi_count' in existing_activity_cols:
    agg_exprs.append(F.sum(F.coalesce(col("total_poi_count"), lit(0))).cast("long").alias("total_poi_count"))

if 'human_activity_index' in existing_activity_cols:
    agg_exprs.append(F.avg(F.coalesce(col("human_activity_index"), lit(0))).alias("human_activity_index"))

if urbanity_col:
    agg_exprs.append(F.first(col(urbanity_col)).alias("urbanity"))

agg_exprs.extend([
    F.count("h3_cell_id").alias("h3_cell_count"),
    F.first("geometry").alias("geometry")
])

# Aggregate by candidate
candidate_features_agg = candidate_with_features.groupBy(
    "candidate_id", "latitude", "longitude", "store_type",
    "city", "state", "drive_time_minutes", "area_sqkm"
).agg(*agg_exprs)

print(f"Aggregated features for {candidate_features_agg.count():,} candidates")

## Exclude Existing Store Trade Areas (Cannibalization Prevention)

In [ ]:
# Exclude candidates whose centroid overlaps with existing LCE store trade areas
try:
    lce_isochrones = spark.table(lce_isochrones_table)
    
    existing_h3_cells = lce_isochrones.select(
        explode(expr("h3_polyfillash3string(ST_AsBinary(geometry), 8)")).alias("covered_h3")
    ).distinct()
    
    print(f"H3 cells covered by existing LCE stores: {existing_h3_cells.count():,}")
    
    candidates_with_h3 = candidate_features_agg.withColumn(
        "centroid_h3", expr("h3_longlatash3string(longitude, latitude, 8)")
    )
    
    candidates_filtered = candidates_with_h3.join(
        existing_h3_cells,
        candidates_with_h3["centroid_h3"] == existing_h3_cells["covered_h3"],
        "left_anti"
    ).drop("centroid_h3")
    
    excluded_count = candidate_features_agg.count() - candidates_filtered.count()
    print(f"Candidates excluded (in existing trade areas): {excluded_count:,}")
    print(f"Candidates remaining: {candidates_filtered.count():,}")
    
except Exception as e:
    print(f"Could not exclude existing trade areas: {e}")
    candidates_filtered = candidate_features_agg

## Write to Gold

In [ ]:
# Add processing timestamp
candidates_final = candidates_filtered.withColumn(
    "processing_timestamp", F.current_timestamp()
)

# Fill nulls in numeric columns
numeric_cols = [
    field.name for field in candidates_final.schema.fields 
    if field.dataType.typeName() in ['long', 'double', 'integer', 'float']
    and field.name not in ['latitude', 'longitude', 'drive_time_minutes', 'area_sqkm']
]
candidates_final = candidates_final.fillna(0, subset=numeric_cols)

print(f"Records to write: {candidates_final.count():,}")
print(f"Columns: {candidates_final.columns}")

# Write to gold
(
    candidates_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"\n✓ Written to {output_table}")

## Summary Statistics

In [ ]:
print("Candidate Features Summary:")
display(spark.sql(f"""
  SELECT
    COUNT(*) as total_candidates,
    ROUND(AVG(population), 0) as avg_population,
    ROUND(AVG(target_demographic_total), 0) as avg_target_demo,
    ROUND(AVG(total_poi_count), 0) as avg_poi_count,
    ROUND(AVG(human_activity_index), 2) as avg_activity_index,
    ROUND(AVG(h3_cell_count), 0) as avg_h3_cells
  FROM {output_table}
"""))

print("\nBy State:")
display(spark.sql(f"""
  SELECT state, COUNT(*) as count, ROUND(AVG(population), 0) as avg_pop
  FROM {output_table}
  GROUP BY state ORDER BY state
"""))

print("\nBy Urbanity:")
display(spark.sql(f"""
  SELECT urbanity, COUNT(*) as count, ROUND(AVG(population), 0) as avg_pop
  FROM {output_table}
  GROUP BY urbanity ORDER BY count DESC
"""))